# AgentRegistry, end to end: one agent on kagent **and** AWS Bedrock AgentCore

**Persona:** a **developer** taking an agent from nothing to running in production. Each cell runs a real command and shows its output; the markdown before it says *what* and *why*.

**The thesis:** an agent on a laptop is a science project. An agent an organisation can *find, govern, and run anywhere* is a platform — it needs a **catalog** of what exists, **packaging** that's reproducible, and a **runtime** that hosts it behind real auth. That's AgentRegistry, driven by the **`arctl`** CLI. The punchline: we publish one agent (plus an MCP tool server and a reusable skill) and deploy **the same catalog agent to two runtimes** — Solo Enterprise for **kagent** and **AWS Bedrock AgentCore** — by changing **one line**, the Deployment's `runtimeRef`.

```mermaid
flowchart LR
  Dev[developer · arctl] -->|init / build / apply| Cat[(AgentRegistry catalog<br/>summarizer · textkit · summary-style)]
  Cat -->|runtimeRef: kind-kagent| K[kagent on kind<br/>OIDC · Anthropic]
  Cat -->|runtimeRef: aws-agentcore| A[AWS Bedrock AgentCore<br/>native Bedrock Claude]
  classDef c fill:#7C3AED,color:#fff,stroke:#5B21B6,stroke-width:2px
  classDef r fill:#D1FAE5,stroke:#10B981,color:#064E3B,stroke-width:2px
  class Cat c
  class K,A r
```


## Connect to the platform

Load the engineer's credentials, put `arctl` on the path, and mint a token for the catalog. `arctl get runtimes` confirms we're talking to the live control plane.

In [ ]:
set -a
[ -f .env.local ] && . ./.env.local
[ -n "${SECRETS_FILE:-}" ] && [ -f "$SECRETS_FILE" ] && . "$SECRETS_FILE"
set +a
export PATH="$HOME/.arctl/bin:$PATH"
export CLUSTER_NAME="${CLUSTER_NAME:-agentcore-demo}"
export ARCTL_API_BASE_URL="${ARCTL_API_BASE_URL:-http://localhost:12121}"
export ARCTL_API_TOKEN="$(curl -s -X POST "$ARCTL_API_BASE_URL/api/autoauth/oauth/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  -d 'grant_type=client_credentials&client_id=admin&scope=openid profile email Groups' | jq -r .access_token)"
echo "arctl $(arctl version 2>/dev/null | awk '/arctl version/{print $3}') · token $([ -n "$ARCTL_API_TOKEN" ] && echo ok || echo MISSING)"
arctl get runtimes

## 1. Scaffold the building blocks with `arctl init`

**What:** `arctl init` generates a complete, runnable project — source, manifest, Dockerfile, env wiring, tests — per artifact kind. **Why:** that's the day of boilerplate a team doesn't write; framework, language and model are chosen up front and the pieces are wired together.

This demo ships three artifacts, already scaffolded **and** customised, under `artifacts/`:

| Kind | Name | What it is |
|------|------|-----------|
| MCPServer | `acme/textkit` | tools `word_count`, `extract_links` (FastMCP, Python) |
| Skill | `summary-style` | a `SKILL.md` house format, baked into the agent at build |
| Agent | `summarizer` | ADK Python, Claude, uses textkit + the skill |

Run this to watch `arctl init` build a fresh agent from nothing (scratch dir), then we use the committed, customised ones.

In [ ]:
cat <<'CMDS'
  arctl init mcp   acme/textkit  --framework fastmcp --language python
  arctl init skill summary-style
  arctl init agent summarizer    --framework adk --language python \
      --model-provider anthropic --model-name claude-haiku-4-5 --local-mcp ./textkit
CMDS
echo
rm -rf /tmp/arctl-scratch && mkdir -p /tmp/arctl-scratch
( cd /tmp/arctl-scratch && arctl init agent demoagent \
    --framework adk --language python \
    --model-provider anthropic --model-name claude-haiku-4-5 >/dev/null )
echo 'arctl generated:'
find /tmp/arctl-scratch/demoagent -maxdepth 2 -type f | sed 's#/tmp/arctl-scratch/##' | sort

**Make the scaffold real.** A tool is a decorated function the loader discovers by filename; the skill is instructions-as-an-artifact; the agent folds the skill into its instruction and reads MCP tools from env. The agent's model is chosen from `MODEL_PROVIDER`, so the *same* code runs on kagent (Anthropic via LiteLLM) and on AgentCore (native Bedrock). Peek:

In [ ]:
echo '===== a textkit tool (one file = one tool) =====';        sed -n '1,20p' artifacts/textkit/src/tools/word_count.py
echo; echo '===== the skill (house format) =====';               sed -n '1,18p' artifacts/summary-style/SKILL.md
echo; echo '===== the agent: model picked from MODEL_PROVIDER ====='; sed -n '/def create_model/,/LiteLlm(model/p' artifacts/summarizer/summarizer/agent.py

## 2. Prove it locally with `arctl run` (no cluster)

**What/why:** before any cluster, `arctl run` is the inner dev loop — it builds the agent image, starts it, and drops you into an interactive A2A chat with the MCP server alongside. The full agent→tools path on Docker alone. It's interactive, so run it in a **terminal**:

```sh
./scripts/test-local.sh
# then:  summarize this: <paste a paragraph with a couple of https:// links>
```

## 3. Build the images and publish to the catalog

**What:** `arctl build --push` builds each artifact's Dockerfile and pushes to the local registry; `arctl apply` registers it in the catalog. **Why:** from here the catalog is the shared source of truth — discoverable, reproducible, each entry resolving to a real OCI image. Order matters once: the MCPServer must exist before the Agent that references it.

In [ ]:
arctl build ./artifacts/textkit    --push   # -> localhost:5001/textkit:latest
arctl build ./artifacts/summarizer --push   # -> localhost:5001/summarizer:latest

In [ ]:
arctl apply -f artifacts/textkit/mcp.yaml
arctl apply -f artifacts/summary-style/skill.yaml
arctl apply -f artifacts/summarizer/agent.yaml
echo; echo '===== the catalog ====='
arctl get mcp acme/textkit; arctl get skill summary-style; arctl get agent summarizer

## 4. Point the registry at the cluster — a Kubernetes **Runtime**

**What:** a `Runtime` is the registry's pointer to somewhere agents can run; for kagent it's a namespace + a kubeconfig. **Why the extra line:** the daemon runs *inside* Docker, so we join it to the kind network and hand it the cluster's **internal** kubeconfig (API server = the control-plane container's hostname). On a real cluster the kubeconfig just works.

In [ ]:
DAEMON_CTR=$(docker ps --filter publish=12121 --format '{{.Names}}' | head -1)
docker network connect kind "$DAEMON_CTR" 2>/dev/null || true
cat > /tmp/runtime-kagent.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata:
  name: kind-kagent
spec:
  type: Kubernetes
  config:
    namespace: kagent
    kubeconfig: |
$(kind get kubeconfig --internal --name "$CLUSTER_NAME" | sed 's/^/      /')
EOF
arctl apply -f /tmp/runtime-kagent.yaml
arctl get runtimes

## 5. Deploy the agent onto kagent (runtime #1)

**What:** a `Deployment` binds an Agent to a Runtime and carries per-instance env. Note `runtimeRef.name: kind-kagent` — **this is the one line that changes for AWS later.** The registry's Kubernetes adapter translates it into kagent CRDs (a BYO Agent + a kmcp MCPServer) and the controller schedules them. The deployment passes `ANTHROPIC_API_KEY`; the agent defaults to the Anthropic/LiteLLM model here.

In [ ]:
echo '===== the Deployment ====='; cat yaml/deployment.yaml
echo; envsubst < yaml/deployment.yaml | arctl apply -f -
echo 'waiting for the textkit kmcp MCPServer...'
for _ in $(seq 1 40); do
  MCPSRV=$(kubectl --context kind-$CLUSTER_NAME -n kagent get mcpserver -o name 2>/dev/null | grep -i textkit | head -1)
  [ -n "$MCPSRV" ] && break; sleep 3
done
[ -n "$MCPSRV" ] && kubectl --context kind-$CLUSTER_NAME -n kagent patch "$MCPSRV" --type merge \
  -p '{"spec":{"deployment":{"cmd":"python","args":["src/main.py"]}}}' && echo "set $MCPSRV cmd"
arctl get deployments

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get agents,pods | grep -vi kmcp-enterprise

## 6. Talk to the hosted agent — through real OIDC

**Why it's honest:** the agent sits behind Solo Enterprise for kagent's OIDC interceptor. `ask.sh` mints a real Keycloak token for **alice** (group `field-fte` → kagent Admin), then POSTs an A2A `message/send`. In the reply: the bold headline + bullets come from the *skill*, the proportional length from the `word_count` *MCP tool*, the `Sources:` line from `extract_links` — every catalog artifact plus the runtime's auth, in one request.

In [ ]:
./scripts/ask.sh "summarize this: AgentRegistry is an open catalog for agents, MCP servers and skills. The arctl CLI scaffolds an artifact, builds it into an OCI image, and publishes it so others can reuse it. A Kubernetes Runtime adapter turns a Deployment into kagent CRDs. Docs at https://aregistry.ai and source at https://github.com/agentregistry-dev/agentregistry."

Optional — open the kagent dashboard: `./scripts/port-forward.sh` then `http://localhost:8080`.

---
# The punchline: the **same** agent on AWS Bedrock AgentCore (runtime #2)

We deploy the identical published `summarizer` to AWS — same catalog entry, no second codebase. We register a `BedrockAgentCore` Runtime and a Deployment whose only meaningful difference is `runtimeRef`. On AWS the agent runs **native Bedrock Claude via the AWS role** (no API key) — the agent picks that model from `MODEL_PROVIDER=bedrock`. **Needs an AWS account; skip for a local-only demo.**

## 7. Sign in to AWS

`AWS_PROFILE` comes from `.env.local`. `aws sso login` opens your browser. The cell also hands those credentials to the arctl daemon (restarts it) so the daemon can assume the cross-account role when it manages AgentCore. Nothing here prints your account or role.

In [ ]:
if [ -z "${AWS_PROFILE:-}" ]; then echo "Set AWS_PROFILE in .env.local (./scripts/setup-env.sh) and re-run the Connect cell."; else
  aws sts get-caller-identity >/dev/null 2>&1 || aws sso login --profile "$AWS_PROFILE"
  export AWS_REGION="${AWS_REGION:-us-east-1}"
  export AWS_ACCOUNT_ID="$(aws sts get-caller-identity --query Account --output text)"
  echo "AWS session live — account ****${AWS_ACCOUNT_ID: -4} / region $AWS_REGION"
  # The daemon (a container) assumes the cross-account role, so give it creds and
  # restart it (the catalog persists in postgres across the restart).
  eval "$(aws configure export-credentials --format env)"
  export AWS_ACCESS_KEY_ID AWS_SECRET_ACCESS_KEY AWS_SESSION_TOKEN
  export DOCKER_REPO="${DOCKER_REPO:-solo-public/agentregistry-enterprise}" OIDC_AUTO_AUTH_ENABLED=true
  arctl daemon stop >/dev/null 2>&1; arctl daemon start >/dev/null 2>&1
  DC=$(docker ps --filter publish=12121 --format '{{.Names}}' | head -1)
  docker network connect kind "$DC" 2>/dev/null || true; sleep 5
  export ARCTL_API_TOKEN="$(curl -s -X POST "$ARCTL_API_BASE_URL/api/autoauth/oauth/token" -H 'Content-Type: application/x-www-form-urlencoded' -d 'grant_type=client_credentials&client_id=admin&scope=openid profile email Groups' | jq -r .access_token)"
  echo 'daemon restarted with AWS credentials'
fi

## 8. Grant AgentRegistry access to your AWS account

`arctl runtime setup bedrock-agent-core` generates a CloudFormation template that creates a cross-account IAM role (so AgentRegistry can manage AgentCore in your account) plus an **External ID**. This makes **no AWS changes** — it prints the template + ID. We then deploy the stack and read back the role ARN.

In [ ]:
mkdir -p .agentcore
arctl runtime setup bedrock-agent-core --aws-account-id "$AWS_ACCOUNT_ID" \
  --role-name AgentRegistryAccessRole-agentcore-demo \
  2> >(tee .agentcore/setup.stderr >&2) > .agentcore/cf.yaml
export AWS_EXTERNAL_ID=$(grep -ioE 'External ID:[[:space:]]*[A-Za-z0-9_-]+' .agentcore/setup.stderr | awk '{print $NF}' | head -1)
echo "External ID parsed: $([ -n "$AWS_EXTERNAL_ID" ] && echo yes || echo NO) · CF template $(wc -l < .agentcore/cf.yaml) lines"

In [ ]:
if aws cloudformation describe-stacks --stack-name AgentRegistryAccess >/dev/null 2>&1; then
  echo 'stack already exists — reusing'
else
  aws cloudformation create-stack --stack-name AgentRegistryAccess \
    --template-body file://.agentcore/cf.yaml --capabilities CAPABILITY_NAMED_IAM
  echo 'waiting for stack create...'; aws cloudformation wait stack-create-complete --stack-name AgentRegistryAccess
fi
export AWS_ROLE_ARN=$(aws cloudformation describe-stacks --stack-name AgentRegistryAccess \
  --query 'Stacks[0].Outputs[?OutputKey==`RoleArn`].OutputValue' --output text)
echo "role ARN: ${AWS_ROLE_ARN%%:role*}:role/****"

## 9. Register the AgentCore **Runtime**

The second `Runtime` — same kind of object as the kagent one, just `type: BedrockAgentCore` pointing at your AWS account via the role + External ID. `arctl get runtimes` then shows both: one Kubernetes, one AWS.

In [ ]:
cat > /tmp/runtime-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata:
  name: aws-agentcore
spec:
  type: BedrockAgentCore
  config:
    roleArn: "${AWS_ROLE_ARN}"
    externalId: "${AWS_EXTERNAL_ID}"
    region: "${AWS_REGION}"
EOF
arctl apply -f /tmp/runtime-aws.yaml
arctl get runtimes

## 10. Push the image to ECR and point the Agent at it

AgentCore can't pull from `localhost:5001` and clones the agent **source** from git at deploy time. So push the image to **ECR** and re-publish the Agent referencing the ECR image + git repo — now as **`modelProvider: bedrock`** so it runs the AWS-role model. (`--platform linux/amd64` since AgentCore wants amd64.)

In [ ]:
export ECR_HOST="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com"
export ECR_IMAGE="${ECR_HOST}/summarizer:0.0.1"
aws ecr describe-repositories --repository-names summarizer >/dev/null 2>&1 \
  || aws ecr create-repository --repository-name summarizer >/dev/null
aws ecr get-login-password --region "$AWS_REGION" | docker login --username AWS --password-stdin "$ECR_HOST" >/dev/null
arctl build ./artifacts/summarizer --push --platform linux/amd64 --image "$ECR_IMAGE"
echo "pushed $ECR_IMAGE"

In [ ]:
# Re-publish the Agent: ECR image + git source AWS can clone, model on Bedrock.
# Set AGENT_GIT_* in .env.local to a repo/branch/subfolder reachable by AWS.
# If AGENT_GIT_URL is a private github repo, inject a fresh gh token so the
# registry can clone it (kept out of the YAML we print).
: "${AGENT_GIT_URL:?set AGENT_GIT_URL in .env.local (run ./scripts/setup-env.sh)}"
CLONE_URL="$AGENT_GIT_URL"
case "$AGENT_GIT_URL" in
  https://github.com/*)
    if command -v gh >/dev/null 2>&1 && gh auth status >/dev/null 2>&1; then
      slug="${AGENT_GIT_URL#https://github.com/}"; slug="${slug%.git}"
      [ "$(gh repo view "$slug" --json isPrivate -q .isPrivate 2>/dev/null)" = "true" ] \
        && CLONE_URL="https://x-access-token:$(gh auth token)@github.com/${slug}.git"
    fi ;;
esac
cat > /tmp/agent-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Agent
metadata:
  name: summarizer
spec:
  description: Summarizes pasted text in the house format, using textkit MCP tools.
  modelName: us.anthropic.claude-haiku-4-5-20251001-v1:0
  modelProvider: bedrock
  source:
    image: ${ECR_IMAGE}
    repository:
      url: ${CLONE_URL}
      branch: ${AGENT_GIT_BRANCH:-main}
      subfolder: ${AGENT_GIT_SUBFOLDER:-agentregistry-agentcore-kind/artifacts/summarizer}
EOF
arctl apply -f /tmp/agent-aws.yaml

## 11. Deploy the same agent to AgentCore (runtime #2)

**The whole point in one cell.** Same Agent — only `runtimeRef` differs (`aws-agentcore` instead of `kind-kagent`). `MODEL_PROVIDER=bedrock` tells the agent to use native Bedrock Claude via the AWS role (no API key). The cell waits for the AWS runtime to provision.

In [ ]:
cat > /tmp/deployment-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Deployment
metadata:
  name: summarizer-agentcore
spec:
  targetRef:
    kind: Agent
    name: summarizer
  runtimeRef:
    kind: Runtime
    name: aws-agentcore          # <-- the only meaningful change vs runtime #1
  runtimeConfig:
    region: ${AWS_REGION}
  env:
    MODEL_PROVIDER: bedrock        # native Bedrock Claude via the AWS role; no API key
    AWS_REGION: ${AWS_REGION}
EOF
arctl apply -f /tmp/deployment-aws.yaml
echo 'waiting for the AWS runtime to provision (CREATING -> READY, a few minutes)...'
for i in $(seq 1 25); do
  S=$(aws bedrock-agentcore-control list-agent-runtimes --region "$AWS_REGION" 2>/dev/null | jq -r '.agentRuntimes[]?|select(.agentRuntimeName=="summarizer_agentcore")|.status')
  echo "[$i] summarizer_agentcore: ${S:-<none>}"; echo "$S" | grep -qiE 'READY|FAILED' && break; sleep 30
done
arctl get deployments

## 12. Test the agent on AgentCore

Invoke the AWS-hosted runtime directly with a JSON-RPC `message/send` and print the reply. Same catalog agent, same skill + tools, now answering from AWS — the only thing that changed between runtime #1 and #2 was the Deployment's `runtimeRef`.

In [ ]:
ARN=$(aws bedrock-agentcore-control list-agent-runtimes --region "$AWS_REGION" | jq -r '.agentRuntimes[]?|select(.agentRuntimeName=="summarizer_agentcore")|.agentRuntimeArn')
PAYLOAD='{"jsonrpc":"2.0","id":"r1","method":"message/send","params":{"message":{"role":"user","messageId":"m-1","parts":[{"kind":"text","text":"summarize this: AgentRegistry is a catalog for agents, MCP servers and skills. arctl scaffolds, builds and publishes them so teams can reuse them. Docs at https://aregistry.ai and source at https://github.com/agentregistry-dev/agentregistry."}]}}}'
aws bedrock-agentcore invoke-agent-runtime --region "$AWS_REGION" --cli-binary-format raw-in-base64-out \
  --agent-runtime-arn "$ARN" --content-type application/json --accept application/json \
  --payload "$PAYLOAD" /tmp/ac-out.json >/dev/null
python3 - <<'PY'
import json
d=json.load(open('/tmp/ac-out.json')); seen=[]
def w(o):
    if isinstance(o,dict):
        if o.get('role')=='user': return
        if o.get('kind')=='text' and isinstance(o.get('text'),str):
            t=o['text'].strip()
            if t and t not in seen: seen.append(t)
        [w(v) for v in o.values()]
    elif isinstance(o,list): [w(v) for v in o]
if 'error' in d: print('ERROR:', json.dumps(d['error']))
else:
    w(d); print('\n\n'.join(seen) if seen else json.dumps(d)[:1200])
PY

**The takeaway:** one catalog entry, governed in one place, ran unchanged on Kubernetes *and* on AWS-managed infrastructure. As a developer, moving or multi-homing an agent is a one-line `runtimeRef` change, not a porting project.

## Teardown

```sh
./scripts/cleanup.sh agentcore   # AWS only
./scripts/cleanup.sh             # everything (cluster, daemon, registry, AWS)
```

In [ ]:
# Uncomment to tear everything down:
# ./scripts/cleanup.sh